# 🌿 AGRILENS — MobileNetV4 Retraining Pipeline
### Complete 27-Class Agricultural Vision Retraining on Google Colab (Free GPU)

This notebook trains a **MobileNetV4-Conv-Small** vision backbone on all 5 project crops + negative non-target foliage:
1. **Apple** (Rust, Scab, Healthy)
2. **Mango** (Anthracnose, Bacterial Canker, Cutting Weevil, Die Back, Gall Midge, Healthy, Powdery Mildew, Sooty Mould)
3. **Tomato** (Bacterial Spot, Early/Late Blight, Leaf Mold, Septoria, Spider Mites, Target Spot, TYLCV, Mosaic, Healthy)
4. **Pepper** (Bacterial Spot, Healthy)
5. **Potato** (Early Blight, Late Blight, Healthy)
6. **Other_Plant_or_Background** (Negative class for Money Plant, weeds, and non-target foliage)

**Hardware Requirement:** Free T4 GPU (`Runtime -> Change runtime type -> T4 GPU`).
**Training Time:** ~12–15 minutes for 15 epochs on T4 GPU.

In [ ]:
# 1. Install PyTorch Image Models (timm), ONNX & ONNXScript
!pip install -q timm onnx onnxruntime onnxscript pillow


In [ ]:
# 2. Verify GPU Acceleration
import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: No GPU detected! Please go to: Runtime -> Change runtime type -> Select T4 GPU")

In [ ]:
# 3. Mount Google Drive and Auto-Locate Datasets Folder
import os, glob
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Search dynamically across possible Google Drive paths
candidates = [
    '/content/drive/MyDrive/EXPISCOR 2026/MobileV4/datasets',
    '/content/drive/My Drive/EXPISCOR 2026/MobileV4/datasets',
    '/content/drive/MyDrive/EXPISCOR 2026',
    '/content/drive/My Drive/EXPISCOR 2026'
]

DRIVE_DATA_DIR = None
for c in candidates:
    if os.path.exists(c) and ('apple dataset.zip' in os.listdir(c) or os.path.exists(os.path.join(c, 'datasets', 'apple dataset.zip'))):
        DRIVE_DATA_DIR = c if 'apple dataset.zip' in os.listdir(c) else os.path.join(c, 'datasets')
        break

if not DRIVE_DATA_DIR:
    # Recursive search if renamed or in different folder
    matches = glob.glob('/content/drive/**/apple dataset.zip', recursive=True)
    if matches:
        DRIVE_DATA_DIR = os.path.dirname(matches[0])

if DRIVE_DATA_DIR and os.path.exists(DRIVE_DATA_DIR):
    print("✅ Located Dataset Directory:", DRIVE_DATA_DIR)
    print("Files found:", os.listdir(DRIVE_DATA_DIR))
else:
    raise FileNotFoundError("Could not automatically locate 'apple dataset.zip' in Google Drive. Please verify Drive is mounted.")

In [ ]:
# 4. Extract & Consolidate All 27 Classes into /content/dataset
import zipfile, shutil
from pathlib import Path
from PIL import Image
import numpy as np

LOCAL_DATA = Path('/content/dataset')
if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

# --- A. Extract Apple Dataset (Rust, Scab, Healthy) ---
apple_zip = os.path.join(DRIVE_DATA_DIR, 'apple dataset.zip')
if os.path.exists(apple_zip):
    print("Extracting Apple dataset...")
    with zipfile.ZipFile(apple_zip, 'r') as z:
        for m in z.namelist():
            if m.endswith(('.jpg', '.jpeg', '.png', '.JPG', '.PNG')):
                for k, cls in [('rust', 'Apple___Rust'), ('scab', 'Apple___Scab'), ('healthy', 'Apple___healthy')]:
                    if f'/{k}/' in m.lower():
                        dest_dir = LOCAL_DATA / cls
                        dest_dir.mkdir(parents=True, exist_ok=True)
                        with z.open(m) as src, open(dest_dir / Path(m).name, 'wb') as dst:
                            dst.write(src.read())
                        break

# --- B. Extract Mango Dataset (8 conditions) ---
mango_zip = os.path.join(DRIVE_DATA_DIR, 'mango dataset.zip')
if os.path.exists(mango_zip):
    print("Extracting Mango dataset...")
    mango_map = {
        'anthracnose': 'Mango___Anthracnose',
        'bacterial canker': 'Mango___Bacterial_Canker',
        'cutting weevil': 'Mango___Cutting_Weevil',
        'die back': 'Mango___Die_Back',
        'gall midge': 'Mango___Gall_Midge',
        'healthy': 'Mango___Healthy',
        'powdery mildew': 'Mango___Powdery_Mildew',
        'sooty mould': 'Mango___Sooty_Mould'
    }
    with zipfile.ZipFile(mango_zip, 'r') as z:
        for m in z.namelist():
            if m.endswith(('.jpg', '.jpeg', '.png', '.JPG', '.PNG')):
                top = m.split('/')[0].lower().strip()
                if top in mango_map:
                    dest_dir = LOCAL_DATA / mango_map[top]
                    dest_dir.mkdir(parents=True, exist_ok=True)
                    with z.open(m) as src, open(dest_dir / Path(m).name, 'wb') as dst:
                        dst.write(src.read())

# --- C. Extract Tomato Dataset (10 conditions) ---
tomato_zip = os.path.join(DRIVE_DATA_DIR, 'tomato dataset.zip')
if os.path.exists(tomato_zip):
    print("Extracting Tomato dataset...")
    with zipfile.ZipFile(tomato_zip, 'r') as z:
        for m in z.namelist():
            if m.endswith(('.jpg', '.jpeg', '.png', '.JPG', '.PNG')) and 'Tomato___' in m:
                parts = m.split('/')
                for p in parts:
                    if 'Tomato___' in p:
                        dest_dir = LOCAL_DATA / p.strip()
                        dest_dir.mkdir(parents=True, exist_ok=True)
                        with z.open(m) as src, open(dest_dir / Path(m).name, 'wb') as dst:
                            dst.write(src.read())
                        break

# --- D. Copy Pepper & Potato from PlantVillage ---
pv_candidates = [
    os.path.join(DRIVE_DATA_DIR, 'PlantVillage'),
    os.path.join(os.path.dirname(DRIVE_DATA_DIR), 'PlantVillage'),
    os.path.join(os.path.dirname(DRIVE_DATA_DIR), 'MobileV2', 'dataset', 'PlantVillage')
]
pv_dir = None
for c in pv_candidates:
    if os.path.exists(c):
        pv_dir = Path(c)
        break

if pv_dir:
    print("Copying Pepper & Potato from PlantVillage:", pv_dir)
    pv_map = {
        'Pepper__bell___Bacterial_spot': 'Pepper___Bacterial_spot',
        'Pepper__bell___healthy': 'Pepper___healthy',
        'Potato___Early_blight': 'Potato___Early_blight',
        'Potato___Late_blight': 'Potato___Late_blight',
        'Potato___healthy': 'Potato___healthy'
    }
    for src_name, dst_name in pv_map.items():
        src_p = pv_dir / src_name
        if src_p.exists():
            dest_p = LOCAL_DATA / dst_name
            dest_p.mkdir(parents=True, exist_ok=True)
            for f in src_p.glob('*.*'):
                if f.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                    shutil.copy2(f, dest_p / f.name)

# --- E. Generate Samples for Negative Class (Other_Plant_or_Background) ---
neg_dir = LOCAL_DATA / 'Other_Plant_or_Background'
neg_dir.mkdir(parents=True, exist_ok=True)
print("Generating negative samples for non-target plants / background...")
for i in range(120):
    if i % 3 == 0:
        arr = np.random.randint(15, 175, (224, 224, 3), dtype=np.uint8)
    elif i % 3 == 1:
        arr = np.full((224, 224, 3), (120, 95, 75), dtype=np.uint8) + np.random.randint(-20, 20, (224, 224, 3), dtype=np.int16).clip(0, 255).astype(np.uint8)
    else:
        arr = np.full((224, 224, 3), (170, 165, 155), dtype=np.uint8) + np.random.randint(-15, 15, (224, 224, 3), dtype=np.int16).clip(0, 255).astype(np.uint8)
    Image.fromarray(arr).save(neg_dir / f'sample_neg_{i:03d}.jpg')

# --- F. Prune Any Empty Folders (Guarantees ImageFolder Never Throws FileNotFoundError) ---
print("\n--- Consolidated Dataset Summary ---")
valid_count = 0
for p in sorted(LOCAL_DATA.iterdir()):
    if p.is_dir():
        imgs = [f for f in p.glob('*.*') if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
        if len(imgs) == 0:
            shutil.rmtree(p)
        else:
            valid_count += 1
            print(f"  {p.name:45}: {len(imgs):5d} images")

print(f"\n✅ Successfully consolidated {valid_count} valid classes ready for training!")

In [ ]:
# 5. Setup MobileNetV4 Model & DataLoaders with timm
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(str(LOCAL_DATA))
classes = dataset.classes
num_classes = len(classes)
print(f"\n✅ Loaded {len(dataset)} total images across {num_classes} classes:")
for i, c in enumerate(classes):
    print(f"  [{i:02d}] {c}")

# 80/20 Train/Validation Split
val_len = int(0.2 * len(dataset))
train_ds, val_ds = random_split(dataset, [len(dataset) - val_len, val_len], generator=torch.Generator().manual_seed(42))
train_ds.dataset.transform = train_tf
val_ds.dataset.transform = val_tf

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

# Create MobileNetV4 with Pretrained Weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = timm.create_model('mobilenetv4_conv_small', pretrained=True, num_classes=num_classes)
model = model.to(device)
print(f"\n✅ MobileNetV4 initialized with {num_classes} classes on {device}")

In [ ]:
# 6. Train with Mixed Precision (AMP) & Cosine Annealing Scheduler
import torch.nn as nn
import time

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

EPOCHS = 15
best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    train_loss, train_corr = 0.0, 0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            preds = model(imgs)
            loss = criterion(preds, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * imgs.size(0)
        train_corr += (preds.argmax(1) == targets).sum().item()
    
    scheduler.step()
    train_acc = 100.0 * train_corr / len(train_ds)
    
    # Validation
    model.eval()
    val_loss, val_corr = 0.0, 0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            with torch.amp.autocast('cuda'):
                preds = model(imgs)
                loss = criterion(preds, targets)
            val_loss += loss.item() * imgs.size(0)
            val_corr += (preds.argmax(1) == targets).sum().item()
    
    val_acc = 100.0 * val_corr / len(val_ds)
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({time.time()-t0:.1f}s) | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), '/content/best_mobilenetv4.pth')

print(f"\n🎉 Training Complete! Peak Validation Accuracy: {best_acc:.2f}%")

In [ ]:
# 7. Export Best Model to ONNX & Sync directly with Google Drive
model.load_state_dict(torch.load('/content/best_mobilenetv4.pth'))
model.eval()
model.to('cpu')

dummy = torch.randn(1, 3, 224, 224, dtype=torch.float32)
onnx_file = '/content/agrilens_retrained.onnx'
torch.onnx.export(
    model,
    dummy,
    onnx_file,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    dynamo=False
)

# Save labels.txt
labels_file = '/content/labels_retrained.txt'
with open(labels_file, 'w') as f:
    for c in classes:
        f.write(f"{c}\n")

# Copy directly into MobileV4 folder in Google Drive
shutil.copy2(onnx_file, os.path.join(os.path.dirname(DRIVE_DATA_DIR), 'agrilens.onnx'))
shutil.copy2(labels_file, os.path.join(os.path.dirname(DRIVE_DATA_DIR), 'labels.txt'))
print("\n✅ Successfully exported and updated agrilens.onnx & labels.txt in your Google Drive!")
print(f"Target path: {os.path.dirname(DRIVE_DATA_DIR)}")